# DSCI 560 Lab11
Po-Hao Chen <br>
4213309111

## Check GPU and CUDA version

In [ ]:
!nvidia-smi
!nvcc --version

Fri Nov 28 09:53:22 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import files

## Part 1. CPU

In [ ]:
# compile matrix_cpu.c
!gcc matrix_cpu.c -o matrix_cpu -O2

# execute and evaluate
!./matrix_cpu 1024

CPU execution time (N=1024): 3.851889 seconds


## Part 2. GPU

In [ ]:
# copile CUDA progeam - matrix_gpu.cu
!nvcc -arch=sm_75 matrix_gpu.cu -o matrix_gpu

# execute GPU test
!./matrix_gpu 1024
!./matrix_gpu 2048
!./matrix_gpu 4096

Matrix Size N = 1024
GPU execution time (Naïve): 0.006809 seconds
Matrix Size N = 2048
GPU execution time (Naïve): 0.053637 seconds
Matrix Size N = 4096
GPU execution time (Naïve): 0.319735 seconds


## Part 4. Optimized CUDA

In [ ]:
# Compile
!nvcc -arch=sm_75 matrix_gpu_optimized.cu -o matrix_gpu_optimized

# execute
!./matrix_gpu_optimized 1024
!./matrix_gpu_optimized 2048
!./matrix_gpu_optimized 4096

Matrix Size N = 1024 (Optimized Tiled CUDA)
GPU execution time (Tiled Optimization): 0.002347 seconds
Matrix Size N = 2048 (Optimized Tiled CUDA)
GPU execution time (Tiled Optimization): 0.018509 seconds
Matrix Size N = 4096 (Optimized Tiled CUDA)
GPU execution time (Tiled Optimization): 0.288269 seconds


## Part 6. Using cuBLAS Library

In [ ]:
# 1. Compile and connect to cuBLAS library
!nvcc -arch=sm_75 matrix_cublas.cu -lcublas -o matrix_cublas

# 2. execute
!./matrix_cublas 1024
!./matrix_cublas 2048
!./matrix_cublas 4096

Matrix Size N = 1024 (cuBLAS)
GPU execution time (cuBLAS): 0.006268 seconds
Matrix Size N = 2048 (cuBLAS)
GPU execution time (cuBLAS): 0.011274 seconds
Matrix Size N = 4096 (cuBLAS)
GPU execution time (cuBLAS): 0.052778 seconds


## Run All Results using collect_results.py

In [ ]:
!python collect_results.py

Compiling all sources...
   Compiling CPU (matrix_cpu.c)...
   Compiling Naïve GPU (matrix_gpu.cu)...
   Compiling Optimized (matrix_gpu_optimized.cu)...
   Compiling cuBLAS (matrix_cublas.cu)...
Compilation process finished.

🚀 Running tests for CPU...
   N=256: 0.019891 sec
   N=512: 0.224031 sec
   N=1024: 3.280722 sec
   N=2048: 88.443179 sec
   N=4096: Execution Timeout
------------------------------
🚀 Running tests for Naïve GPU...
   N=256: 0.008071 sec
   N=512: 0.007691 sec
   N=1024: 0.007581 sec
   N=2048: 0.007705 sec
   N=4096: 0.010922 sec
------------------------------
🚀 Running tests for Optimized...
   N=256: 0.011163 sec
   N=512: 0.011361 sec
   N=1024: 0.010567 sec
   N=2048: 0.010849 sec
   N=4096: 0.007325 sec
------------------------------
🚀 Running tests for cuBLAS...
   N=256: 0.014966 sec
   N=512: 0.005545 sec
   N=1024: 0.006221 sec
   N=2048: 0.011343 sec
   N=4096: 0.052835 sec
------------------------------

All done! Results saved to 'final_comparison.cs

In [ ]:
files.download('final_comparison.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python plot_matrix_performance.py

Saved plot_log_scale.png (Best for overall comparison)
Saved plot_gpu_only.png (Best for analyzing CUDA optimizations)
Figure(1000x600)
Figure(1000x600)


In [ ]:
files.download('plot_log_scale.png')
files.download('plot_gpu_only.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Part 7. Creating a Shared Library and Using it in Python

In [ ]:
# Compile as a shared library (.so)
!nvcc -arch=sm_75 -Xcompiler -fPIC -shared matrix_lib.cu -o libmatrix.so

# Check if the file is created
!ls -lh libmatrix.so

-rwxr-xr-x 1 root root 985K Nov 28 09:57 libmatrix.so


In [ ]:
files.download('libmatrix.so')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Test my library using python code

In [ ]:
!python test_lib.py

Testing N=1024 with Python + CUDA...
-> Done in 0.2135 seconds

Testing N=2048 with Python + CUDA...
-> Done in 0.0665 seconds

Testing N=4096 with Python + CUDA...
-> Done in 0.4047 seconds



## Part 7.
### Part 2: Adding Custom Functions to the Shared Library

In [ ]:
!nvcc -arch=sm_75 -Xcompiler -fPIC -shared convolution_lib.cu -o libconvolution.so
!ls -lh libconvolution.so

-rwxr-xr-x 1 root root 985K Nov 28 09:57 libconvolution.so


In [ ]:
files.download('libconvolution.so')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python test_convolution.py

=== STARTING MULTI-SIZE PERFORMANCE TEST ===
Testing Image: 800x533 | Filter: 3x3...
Testing Image: 2400x1599 | Filter: 3x3...
Testing Image: 4000x2665 | Filter: 3x3...
Testing Image: 2400x1599 | Filter: 5x5...
Testing Image: 2400x1599 | Filter: 7x7...

Experiment Type      | Image Size      | Filter   | GPU (s)    | CPU (s)    | Speedup 
--------------------------------------------------------------------------------
Varying Image Size   | 800x533         | 3x3      | 0.0017     | 0.0392     | 23.5x
Varying Image Size   | 2400x1599       | 3x3      | 0.0108     | 0.3184     | 29.5x
Varying Image Size   | 4000x2665       | 3x3      | 0.0234     | 0.9023     | 38.5x
Varying Filter Size  | 2400x1599       | 5x5      | 0.0092     | 0.8485     | 91.8x
Varying Filter Size  | 2400x1599       | 7x7      | 0.0102     | 1.0671     | 105.1x

Visual result saved as 'convolution_output_demo.png'


In [ ]:
files.download('convolution_output_demo.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!nvcc -arch=sm_75 convolution_standalone.cu -o convolution_standalone
!./convolution_standalone

Benchmarking Standalone CUDA...
Image Size: 4000 x 2665 (Matching Python Mosaic)
------------------------------------------------
Direct CUDA Executable Time: 0.0014 seconds
------------------------------------------------
